# TorchKM revision campaign, run from a notebook

Run this on a GPU node (a JupyterLab session with one GPU). Each cell is
self-contained; the campaign itself runs in the background so the notebook
never blocks, and every step is resumable: rerunning with the same `OUT`
directory skips finished experiments.

Order: clone → install → **restart the kernel** → check GPU → data → smoke →
launch → monitor → R baselines → tables and figures → commit results.

In [ ]:
import os, sys
ROOT = os.path.expanduser("~/torchkm")
if not os.path.exists(ROOT):
    !git clone --branch claude/great-faraday-id8jaw --single-branch https://github.com/YikaiZhang95/torchkm.git {ROOT}
%cd {ROOT}
!git fetch -q origin claude/great-faraday-id8jaw && git checkout -q claude/great-faraday-id8jaw && git pull -q --ff-only
!git log --oneline -1

## Install into this kernel's environment

Pick the PyTorch wheel matching the driver (`nvidia-smi` prints the CUDA
version it supports): `cu121`, `cu124` or `cu126`. Falkon and cuML are
optional; if either fails to install its column is skipped, nothing else
breaks. ThunderSVM is a source build (see `benchmarks/environment/README.md`).

In [ ]:
TORCH_CUDA = "cu124"
%pip install -q --index-url https://download.pytorch.org/whl/{TORCH_CUDA} torch
%pip install -q -e ".[dev,examples,viz]" nvidia-ml-py psutil matplotlib

In [ ]:
# optional GPU comparison libraries (each may take several minutes)
%pip install -q --no-build-isolation falkon
%pip install -q --extra-index-url=https://pypi.nvidia.com cuml-cu12

**Restart the kernel now** (Kernel → Restart) so the new packages load, then continue.

In [ ]:
import os, sys, torch, torchkm
ROOT = os.path.expanduser("~/torchkm"); os.chdir(ROOT)
PY = sys.executable
print("python:", PY)
print("torch", torch.__version__, "| cuda build", torch.version.cuda, "| cuda available", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("torchkm", torchkm.__version__, torchkm.__file__)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## Data

The loaders read `.bz2`/`.xz` directly, so nothing needs decompressing. If the
GPU node has no internet, run the same command once on the login node.

In [ ]:
DATA = os.path.expanduser("~/libsvm")
!DATA={DATA} bash benchmarks/download_data.sh

## Smoke test on the GPU (a minute)

In [ ]:
for s in ["bench_memory_envelope", "bench_gpu_libraries", "bench_covtype_rank", "bench_kqr", "bench_dwd", "bench_solver_quality"]:
    print("==", s)
    !{PY} benchmarks/{s}.py --smoke --device cuda 2>&1 | grep -v Warning | tail -3
!{PY} -c "import sys; sys.path.insert(0,'benchmarks'); from _libraries import library_availability, ALL_LIBRARIES; [print(f'{k:>16}:', 'ok' if v is None else v) for k, v in library_availability(ALL_LIBRARIES).items()]"

## Launch the campaign in the background

`run_campaign.sh` runs experiments E1 to E10 in dependency order (settings in
`EXPERIMENT_DESIGN.md`) and writes one JSON and one log per step into `OUT`.
Set `ONLY` to run a subset (e.g. `"E8 E1a E5"`). To resume after an
interruption, run this cell again with the same `OUT`.

In [ ]:
import datetime, subprocess
OUT = f"benchmarks/results/{datetime.datetime.utcnow():%Y%m%dT%H%M%SZ}"   # reuse an existing dir to resume
ONLY = ""            # e.g. "E8 E1a E5"; empty = everything
THUNDERSVM = ""      # e.g. os.path.expanduser("~/thundersvm/python") if built
os.makedirs(OUT, exist_ok=True)
env = dict(os.environ, DATA=DATA, OUT=OUT, DEVICE="cuda", PYTHON=PY, ONLY=ONLY)
if THUNDERSVM: env["THUNDERSVM"] = THUNDERSVM
log = open(f"{OUT}/campaign.log", "a")
proc = subprocess.Popen(["bash", "benchmarks/run_campaign.sh"], env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print("started pid", proc.pid, "->", OUT)
print("resume later with OUT =", repr(OUT))

## Monitor

In [ ]:
print("running" if proc.poll() is None else f"finished with code {proc.returncode}")
!tail -n 15 {OUT}/campaign.log
!ls -l {OUT}

In [ ]:
# progress of one step: its own log
!tail -n 5 {OUT}/exact_torchkm.log 2>/dev/null || echo "not started yet"

## R baselines (fastkqr, kernlab, kerndwd)

Needs `Rscript` with the packages from `benchmarks/environment/r-packages.R`.
Runs on the exported splits after E6 and E7 have finished. Check the
argument names against the installed package versions the first time.

In [ ]:
!Rscript benchmarks/environment/r-packages.R 2>&1 | tail -3
!Rscript benchmarks/r/bench_kqr.R {OUT}/kqr_splits {OUT}/kqr_r.csv 2>&1 | tail -5
!Rscript benchmarks/r/bench_dwd.R {OUT}/dwd_splits {OUT}/dwd_r.csv 2>&1 | tail -5

## Tables and figures

In [ ]:
import glob
for f in sorted(glob.glob(f"{OUT}/*.json")):
    print("\n###", os.path.basename(f))
    !{PY} benchmarks/make_tables.py {f}
!{PY} benchmarks/make_tables.py {OUT}/kqr.json --r-csv {OUT}/kqr_r.csv 2>/dev/null
!{PY} benchmarks/make_figures.py --results {OUT}

In [ ]:
from IPython.display import Image, display
for name in ["fig1_envelope", "fig2_scaling", "fig3_covtype_budget", "fig4_solver_quality"]:
    p = f"{OUT}/figures/{name}.png"
    if os.path.exists(p):
        display(Image(p))

## Commit the archive

Only the JSON, CSV, logs, tables and figures are committed; the exported
splits are ignored by `.gitignore`.

In [ ]:
!git add {OUT} && git -c user.name="Yikai Zhang" -c user.email="skyezhang1995@gmail.com" commit -q -m "Archive campaign results {OUT}" && git push origin claude/great-faraday-id8jaw